# Transfer Learning sur FashionMNIST
## Utilisation de modèles pré-entraînés sur ImageNet : ResNet18, VGG16, EfficientNet-B0

**Objectif du TP :**
- Adapter des modèles pré-entraînés (ImageNet → FashionMNIST)
- Résoudre le problème de canal (1→3) et de taille (28×28 → 224×224)
- Modifier la dernière couche (1000 classes → 10 classes)
- Geler des couches (Transfer Learning)
- Comparer les performances des différents modèles

## 1. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

# Vérifier si GPU disponible
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Appareil utilisé : {device}")

# Classes FashionMNIST
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f"Nombre de classes : {len(classes)}")

## 2. Problème principal et solution

### Incompatibilités entre FashionMNIST et ImageNet

| Propriété | FashionMNIST | ImageNet (attendu) |
|-----------|-------------|---------------------|
| Canaux | 1 (gris) | 3 (RGB) |
| Taille | 28×28 | 224×224 |
| Normalisation | [0, 1] | mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225] |
| Classes | 10 | 1000 |

### Solution : Pipeline de transformations

## 3. Chargement des données avec transformations adaptées

In [ ]:
# ─── Transformations pour les modèles pré-entraînés ───────────────────────────
# On doit :
#   1. Redimensionner 28×28 → 224×224
#   2. Convertir en tensor (donne 1 canal)
#   3. Répliquer le canal gris 3 fois (1→3 canaux) via Lambda
#   4. Normaliser avec les stats ImageNet

imagenet_transform = transforms.Compose([
    transforms.Resize((224, 224)),          # 28×28 → 224×224
    transforms.ToTensor(),                  # PIL → Tensor [0,1], shape (1, 224, 224)
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)),  # (1,H,W) → (3,H,W) : gris → RGB
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         # Moyennes ImageNet par canal
        std=[0.229, 0.224, 0.225]           # Écarts-types ImageNet par canal
    )
])

# Transformation simple (pour référence / CNN custom)
simple_transform = transforms.Compose([
    transforms.ToTensor()
])

# ─── Chargement du dataset ─────────────────────────────────────────────────────
train_data = torchvision.datasets.FashionMNIST(
    root='data', train=True, download=True, transform=imagenet_transform
)
test_data = torchvision.datasets.FashionMNIST(
    root='data', train=False, download=True, transform=imagenet_transform
)

# ─── DataLoaders ──────────────────────────────────────────────────────────────
BATCH_SIZE = 32

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train : {len(train_data)} images  |  {len(train_loader)} batches de {BATCH_SIZE}")
print(f"Test  : {len(test_data)} images  |  {len(test_loader)} batches de {BATCH_SIZE}")

# Vérification de la forme après transformation
img, lbl = train_data[0]
print(f"\nForme d'une image transformée : {img.shape}")  # Attendu : torch.Size([3, 224, 224])
print(f"Label : {lbl} ({classes[lbl]})")

In [ ]:
# ─── Visualisation de quelques exemples ───────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Exemples FashionMNIST (après transformation)", fontsize=14)

for i, ax in enumerate(axes.flat):
    img, lbl = test_data[i]
    # Dénormaliser pour affichage
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = img * std + mean
    img_display = img_display.permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img_display[:,:,0], cmap='gray')  # afficher canal 0 (identique aux 3)
    ax.set_title(classes[lbl], fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Fonctions d'entraînement et d'évaluation

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Entraîne le modèle sur une époque, retourne (loss, accuracy)."""
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    """Évalue le modèle, retourne (loss, accuracy)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def train_model(model, train_loader, test_loader, criterion, optimizer,
                n_epochs, device, model_name="Model"):
    """Boucle complète d'entraînement avec affichage."""
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    print(f"\n{'='*60}")
    print(f"  Entraînement de : {model_name}")
    print(f"{'='*60}")

    for epoch in tqdm(range(1, n_epochs + 1), desc=model_name):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_loss'].append(te_loss)
        history['test_acc'].append(te_acc)

        print(f"Epoch {epoch:2d}/{n_epochs} | "
              f"Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.2f}% | "
              f"Test  Loss: {te_loss:.4f}, Acc: {te_acc:.2f}%")

    print(f"\n✓ Meilleure accuracy test : {max(history['test_acc']):.2f}%")
    return history


def get_all_predictions(model, loader, device):
    """Collecte toutes les prédictions pour le rapport de classification."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


def plot_history(history, model_name):
    """Affiche les courbes loss/accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Courbes d'entraînement — {model_name}", fontsize=13)
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], label='Train')
    ax1.plot(epochs, history['test_loss'],  label='Test')
    ax1.set_xlabel('Époque'); ax1.set_ylabel('Loss')
    ax1.set_title('Loss'); ax1.legend()

    ax2.plot(epochs, history['train_acc'], label='Train')
    ax2.plot(epochs, history['test_acc'],  label='Test')
    ax2.set_xlabel('Époque'); ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Accuracy'); ax2.legend()

    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, classes, model_name):
    """Affiche la matrice de confusion normalisée."""
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')
    ax.set_title(f'Matrice de Confusion — {model_name}')
    plt.tight_layout(); plt.show()

print("✓ Fonctions utilitaires définies")

## 5. Concept de gel de couches (Freezing)

### Pourquoi geler des couches ?

Les modèles pré-entraînés ont déjà appris des **features génériques** (bords, textures, formes) sur ImageNet. En gelant ces couches, on :
- **Conserve** les poids pré-entraînés (bonne initialisation)
- **Accélère** l'entraînement (moins de paramètres à mettre à jour)
- **Évite** le sur-ajustement sur un petit dataset

### Stratégies de gel

```
[Conv Block 1] → [Conv Block 2] → ... → [Conv Block N] → [Classifier]
   GELÉ              GELÉ                   GELÉ           ENTRAÎNÉ ✓
```

On peut aussi ne geler qu'une partie des couches :
```
[Conv Block 1] → [Conv Block 2] → [Conv Block 3] → [Conv Block 4] → [Classifier]
   GELÉ              GELÉ           ENTRAÎNÉ ✓       ENTRAÎNÉ ✓      ENTRAÎNÉ ✓
```

In [ ]:
# ─── Démonstration du gel de couches ──────────────────────────────────────────
def freeze_all(model):
    """Gèle TOUTES les couches du modèle."""
    for param in model.parameters():
        param.requires_grad = False

def unfreeze_all(model):
    """Dégèle toutes les couches."""
    for param in model.parameters():
        param.requires_grad = True

def count_trainable(model):
    """Compte les paramètres entraînables vs total."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Paramètres entraînables : {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return trainable, total

print("Fonctions de gel définies ✓")

## 6. Modèle 1 : ResNet18

**Architecture :** Residual Network — blocs résiduels avec connexions en skip.  
**Dernière couche :** `model.fc` — couche fully-connected (Linear)

In [ ]:
# ─── Chargement ResNet18 pré-entraîné ─────────────────────────────────────────
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("Architecture ResNet18 (résumé) :")
print(resnet18)

In [ ]:
# ─── Étape 1 : Geler TOUTES les couches ──────────────────────────────────────
freeze_all(resnet18)
print("Après gel de toutes les couches :")
count_trainable(resnet18)

# ─── Étape 2 : Modifier la dernière couche (fc) ───────────────────────────────
# ResNet18 : dernière couche = resnet18.fc = Linear(512, 1000)
# On la remplace par Linear(512, 10) — 10 classes FashionMNIST
in_features_resnet = resnet18.fc.in_features  # 512
resnet18.fc = nn.Linear(in_features_resnet, 10)  # La nouvelle couche est entraînable par défaut

print("\nAprès remplacement de la couche fc (1000 → 10 classes) :")
print(f"  Nouvelle couche fc : {resnet18.fc}")
count_trainable(resnet18)
# → seule la couche fc est entraînable

In [ ]:
# ─── Variante : Geler seulement une partie des couches ────────────────────────
# Ici on gèle les couches layer1 et layer2, et on laisse layer3, layer4, fc libres

resnet18_partial = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Geler tout d'abord
freeze_all(resnet18_partial)

# Dégeler les derniers blocs (fine-tuning profond)
for param in resnet18_partial.layer3.parameters():
    param.requires_grad = True
for param in resnet18_partial.layer4.parameters():
    param.requires_grad = True

# Remplacer et dégeler fc
resnet18_partial.fc = nn.Linear(resnet18_partial.fc.in_features, 10)

print("ResNet18 — gel partiel (layer1+layer2 gelés, layer3+layer4+fc libres) :")
count_trainable(resnet18_partial)

In [ ]:
# ─── Entraînement ResNet18 (toutes couches gelées sauf fc) ───────────────────
resnet18 = resnet18.to(device)

criterion  = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(filter(lambda p: p.requires_grad, resnet18.parameters()), lr=1e-3)

N_EPOCHS = 5

history_resnet18 = train_model(
    resnet18, train_loader, test_loader,
    criterion, optimizer_resnet,
    n_epochs=N_EPOCHS, device=device,
    model_name="ResNet18 (fc seul)"
)

In [ ]:
plot_history(history_resnet18, "ResNet18")

# Rapport de classification
y_pred_r18, y_true_r18 = get_all_predictions(resnet18, test_loader, device)
print("\nRapport de classification — ResNet18 :")
print(classification_report(y_true_r18, y_pred_r18, target_names=classes))

plot_confusion_matrix(y_true_r18, y_pred_r18, classes, "ResNet18")

## 7. Modèle 2 : VGG16

**Architecture :** Très profond, blocs de convolutions empilées + 3 couches fully-connected.  
**Dernière couche :** `model.classifier[6]` — 6ème élément du Sequential classifier

In [ ]:
# ─── Chargement VGG16 pré-entraîné ───────────────────────────────────────────
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Voir la structure du classifier
print("Classifier de VGG16 :")
print(vgg16.classifier)
print(f"\nDernière couche (index 6) : {vgg16.classifier[6]}")

In [ ]:
# ─── Gel + Modification ───────────────────────────────────────────────────────
# Geler toutes les couches convolutives (features)
for param in vgg16.features.parameters():
    param.requires_grad = False

print("Après gel des features (couches convolutives) :")
count_trainable(vgg16)

# Remplacer la dernière couche du classifier : Linear(4096, 1000) → Linear(4096, 10)
vgg16.classifier[6] = nn.Linear(4096, 10)

print("\nAprès remplacement classifier[6] (1000 → 10 classes) :")
count_trainable(vgg16)
print(f"Nouvelle dernière couche : {vgg16.classifier[6]}")

In [ ]:
# ─── Entraînement VGG16 ───────────────────────────────────────────────────────
vgg16 = vgg16.to(device)

optimizer_vgg16 = optim.Adam(filter(lambda p: p.requires_grad, vgg16.parameters()), lr=1e-4)

history_vgg16 = train_model(
    vgg16, train_loader, test_loader,
    criterion, optimizer_vgg16,
    n_epochs=N_EPOCHS, device=device,
    model_name="VGG16 (classifier seul)"
)

In [ ]:
plot_history(history_vgg16, "VGG16")

y_pred_vgg, y_true_vgg = get_all_predictions(vgg16, test_loader, device)
print("\nRapport de classification — VGG16 :")
print(classification_report(y_true_vgg, y_pred_vgg, target_names=classes))

plot_confusion_matrix(y_true_vgg, y_pred_vgg, classes, "VGG16")

## 8. Modèle 3 : EfficientNet-B0

**Architecture :** Réseau optimisé par Neural Architecture Search — meilleur compromis taille/performance.  
**Dernière couche :** `model.classifier[1]` — Linear dans le Sequential classifier

In [ ]:
# ─── Chargement EfficientNet-B0 pré-entraîné ─────────────────────────────────
efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

print("Classifier EfficientNet-B0 :")
print(efficientnet.classifier)
print(f"\nDernière couche (index 1) : {efficientnet.classifier[1]}")

In [ ]:
# ─── Gel + Modification ───────────────────────────────────────────────────────
# Geler toutes les couches features
for param in efficientnet.features.parameters():
    param.requires_grad = False

print("Après gel des features :")
count_trainable(efficientnet)

# Remplacer la dernière couche : Linear(1280, 1000) → Linear(1280, 10)
in_features_eff = efficientnet.classifier[1].in_features  # 1280
efficientnet.classifier[1] = nn.Linear(in_features_eff, 10)

print("\nAprès remplacement classifier[1] (1000 → 10 classes) :")
count_trainable(efficientnet)

In [ ]:
# ─── Gel partiel : Fine-tuning des 2 derniers blocs ─────────────────────────
# EfficientNet a des MBConv blocks dans features[0..8]
# On dégèle les derniers blocs pour un fine-tuning plus profond

# Dégeler les 2 derniers blocs de features (indices 7 et 8)
for param in efficientnet.features[7].parameters():
    param.requires_grad = True
for param in efficientnet.features[8].parameters():
    param.requires_grad = True

print("Après dégel partiel (features[7] et features[8] libres) :")
count_trainable(efficientnet)

In [ ]:
# ─── Entraînement EfficientNet-B0 ────────────────────────────────────────────
efficientnet = efficientnet.to(device)

optimizer_eff = optim.Adam(filter(lambda p: p.requires_grad, efficientnet.parameters()), lr=1e-3)

# Scheduler pour réduire le LR si plateau
scheduler_eff = optim.lr_scheduler.ReduceLROnPlateau(optimizer_eff, mode='max', patience=2, factor=0.5)

history_eff = train_model(
    efficientnet, train_loader, test_loader,
    criterion, optimizer_eff,
    n_epochs=N_EPOCHS, device=device,
    model_name="EfficientNet-B0"
)

In [ ]:
plot_history(history_eff, "EfficientNet-B0")

y_pred_eff, y_true_eff = get_all_predictions(efficientnet, test_loader, device)
print("\nRapport de classification — EfficientNet-B0 :")
print(classification_report(y_true_eff, y_pred_eff, target_names=classes))

plot_confusion_matrix(y_true_eff, y_pred_eff, classes, "EfficientNet-B0")

## 9. Comparaison des modèles

In [ ]:
# ─── Tableau comparatif ───────────────────────────────────────────────────────
results = {
    'ResNet18':       history_resnet18,
    'VGG16':          history_vgg16,
    'EfficientNet-B0': history_eff,
}

print(f"{'Modèle':<20} {'Train Acc max':<18} {'Test Acc max':<15} {'Params entraînés'}")
print("-" * 75)

model_params = {
    'ResNet18':        resnet18,
    'VGG16':           vgg16,
    'EfficientNet-B0': efficientnet,
}

for name, hist in results.items():
    tr_max = max(hist['train_acc'])
    te_max = max(hist['test_acc'])
    trainable, total = count_trainable(model_params[name])
    print(f"{name:<20} {tr_max:<18.2f} {te_max:<15.2f} {trainable:,} / {total:,}")

In [ ]:
# ─── Graphique comparatif ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparaison des modèles pré-entraînés sur FashionMNIST", fontsize=13)

colors = {'ResNet18': 'blue', 'VGG16': 'orange', 'EfficientNet-B0': 'green'}

for name, hist in results.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    color = colors[name]
    ax1.plot(epochs, hist['test_loss'], label=name, color=color)
    ax2.plot(epochs, hist['test_acc'],  label=name, color=color)

ax1.set_xlabel('Époque'); ax1.set_ylabel('Loss (test)')
ax1.set_title('Test Loss'); ax1.legend()

ax2.set_xlabel('Époque'); ax2.set_ylabel('Accuracy % (test)')
ax2.set_title('Test Accuracy'); ax2.legend()

plt.tight_layout()
plt.show()

## 10. Récapitulatif — Points clés du TP

### Adaptations nécessaires

| Problème | Solution PyTorch |
|----------|-----------------|
| 1 canal → 3 canaux | `transforms.Lambda(lambda x: x.repeat(3, 1, 1))` |
| 28×28 → 224×224 | `transforms.Resize((224, 224))` |
| Normalisation ImageNet | `transforms.Normalize(mean=[...], std=[...])` |
| 1000 classes → 10 classes | Remplacer la dernière couche |

### Comment geler des couches

```python
# Geler tout le modèle
for param in model.parameters():
    param.requires_grad = False

# Remplacer la dernière couche (automatiquement entraînable)
model.fc = nn.Linear(in_features, 10)          # ResNet
model.classifier[6] = nn.Linear(4096, 10)      # VGG16
model.classifier[1] = nn.Linear(1280, 10)      # EfficientNet

# N'optimiser que les paramètres entraînables
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
```

### Gel partiel (fine-tuning avancé)

```python
freeze_all(model)                                   # Geler tout
for param in model.layer4.parameters():            # Dégeler les derniers blocs
    param.requires_grad = True
model.fc = nn.Linear(512, 10)                       # Nouvelle tête
```